1. You are given a large NumPy array of size 5000000 initialized with random
values. Compute the following element-wise operation: f(x)=x2+3x+5, for
each element in the array and convert it into a CUDA kernel using Numba.
Compare performance diﬀerence of CPU with GPU.

In [15]:
import numpy as np
from numba import cuda
import time

N = 5_000_000
arr = np.random.rand(N).astype(np.float32)  # FP32

def cpu_compute(arr):
    return arr**2 + 3*arr + 5

@cuda.jit
def gpu_compute(arr, result):
    idx = cuda.grid(1)
    if idx < arr.size:
        x = arr[idx]
        result[idx] = x*x + 3*x + 5

# CPU
start = time.time()
cpu_result = cpu_compute(arr)
cpu_time = time.time() - start



d_arr = cuda.to_device(arr)
d_result = cuda.device_array_like(arr)

threads_per_block = 256
blocks = (N + threads_per_block - 1) // threads_per_block


# GPU Warmup + Numba First Run
gpu_compute[blocks, threads_per_block](d_arr, d_result)
cuda.synchronize()


# GPU
start = time.time()
gpu_compute[blocks, threads_per_block](d_arr, d_result)
cuda.synchronize()
gpu_time = time.time() - start

result = d_result.copy_to_host()


print("CPU Time:", cpu_time)
print("GPU Time (after warm-up):", gpu_time)
print("Speedup:", cpu_time / gpu_time)

CPU Time: 0.017918109893798828
GPU Time (after warm-up): 0.0008001327514648438
Speedup: 22.393921334922528


In [7]:
import numpy as np
from numba import cuda
import time

N = 5_000_000
arr = np.random.rand(N).astype(np.float64)  # FP64

def cpu_compute(arr):
    return arr**2 + 3*arr + 5

@cuda.jit
def gpu_compute(arr, result):
    idx = cuda.grid(1)
    if idx < arr.size:
        x = arr[idx]
        result[idx] = x*x + 3*x + 5

# CPU
start = time.time()
cpu_result = cpu_compute(arr)
cpu_time = time.time() - start



d_arr = cuda.to_device(arr)
d_result = cuda.device_array_like(arr)

threads_per_block = 256
blocks = (N + threads_per_block - 1) // threads_per_block


# GPU Warmup + Numba First Run
gpu_compute[blocks, threads_per_block](d_arr, d_result)
cuda.synchronize()


# GPU
start = time.time()
gpu_compute[blocks, threads_per_block](d_arr, d_result)
cuda.synchronize()
gpu_time = time.time() - start

result = d_result.copy_to_host()


print("CPU Time:", cpu_time)
print("GPU Time (after warm-up):", gpu_time)
print("Speedup:", cpu_time / gpu_time)

CPU Time: 0.037830352783203125
GPU Time (after warm-up): 0.0005795955657958984
Speedup: 65.2702591526121


2. Implement and benchmark a 1-D histogram computation for 1 million
random values in Python using Numba. Compare diﬀerent approaches (pure
Python, NumPy, and Numba-accelerated) and analyze performance and
correctness.

In [19]:
import numpy as np
import time
from numba import njit

N = 1_000_000
bins = 256
data = np.random.randint(0, bins, N)


def histogram_python(data, bins):
    hist = [0] * bins
    for x in data:
        hist[int(x)] += 1
    return hist

def histogram_numpy(data, bins):
    hist, _ = np.histogram(data, bins=bins, range=(0, bins))
    return hist

@njit
def histogram_numba(data, bins):
    hist = np.zeros(bins, dtype=np.int64)
    for i in range(len(data)):
        hist[data[i]] += 1
    return hist


# Python
start = time.time()
hist_py = histogram_python(data, bins)
time_py = time.time() - start

# NumPy
start = time.time()
hist_np = histogram_numpy(data, bins)
time_np = time.time() - start

# Numba
hist_nb = histogram_numba(data, bins)

# Numba
start = time.time()
hist_nb = histogram_numba(data, bins)
time_nb = time.time() - start


print("Execution Times:")
print(f"Python  : {time_py} sec")
print(f"NumPy   : {time_np} sec")
print(f"Numba   : {time_nb} sec")

print("\nSpeedups:")
print(f"Numba vs Python: {time_py / time_nb}x")
print(f"NumPy vs Python: {time_py / time_np}x")

Execution Times:
Python  : 0.1368710994720459 sec
NumPy   : 0.013280153274536133 sec
Numba   : 0.0011134147644042969 sec

Speedups:
Numba vs Python: 122.92912205567451x
NumPy vs Python: 10.306439740758695x


3. Write a function monte_carlo_pi(nsamples) that estimates the value of π by
generating random x, y coordinates between 0 and 1 and checking if they fall
inside a unit circle (x2 + y2 < 1).

a. Implement the function in pure Python first and later create a Numba
version.

b. Program a script to compare the execution time for 5 million samples.
Report the Speedup Factor (Python Time / Numba Time).

c. Why does the very first execution of the Numba func8on take slightly
longer than the second execution?

In [22]:
import random
from numba import njit
import numpy as np
import time

def monte_carlo_pi(nsamples):
    inside = 0
    for _ in range(nsamples):
        x = random.random()
        y = random.random()
        if x*x + y*y < 1:
            inside += 1
    return 4 * inside / nsamples

@njit
def monte_carlo_pi_numba(nsamples):
    inside = 0
    for i in range(nsamples):
        x = np.random.rand()
        y = np.random.rand()
        if x*x + y*y < 1:
            inside += 1
    return 4 * inside / nsamples


N = 5_000_000

start = time.time()
monte_carlo_pi(N)
py_time = time.time() - start

start = time.time()
monte_carlo_pi_numba(N)  # 1st run.
nb_time1 = time.time() - start

# First execution of the Numba function take slightly longer than the
# second execution as it first interprets then compiles the code
# while the second execution simply runs the compiled code from the first run.

start = time.time()
monte_carlo_pi_numba(N) # 2nd run
nb_time2 = time.time() - start

print(f"Python: {py_time} sec")
print(f"Numba (1st run): {nb_time1} sec")
print(f"Numba (2nd run): {nb_time2} sec")
print(f"Speedup: {py_time / nb_time2}x")

Python: 0.8489437103271484 sec
Numba (1st run): 0.14038443565368652 sec
Numba (2nd run): 0.05219674110412598 sec
Speedup: 16.264304865961112x


4. You have a 1D NumPy array representing pixel intensities (values 0–255). You
need to increase the brightness of every pixel by 20%, but ensure no value
exceeds 255.

a. Write a function adjust_brightness(pixel_value) using the @vectorize
decorator.

b. Apply this function to an array of 10 million random integers.

c. Change the decorator to @vectorize(['int64(int64)'], target='parallel').
Measure the time diﬀerence when the work is automatically split
across your CPU cores.

d. What happens if you try to pass a list instead of a NumPy array to this
function?

In [39]:
import numpy as np
from numba import vectorize
import time


@vectorize(['int64(int64)'])
def adjust_brightness_seq(pixel_value):
    val = int(pixel_value * 1.2)
    return val if val <= 255 else 255

@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_parallel(pixel_value):
    val = int(pixel_value * 1.2)
    return val if val <= 255 else 255

pixels = np.random.randint(0, 256, 10_000_000)


# warmup
adjust_brightness_seq(pixels)
adjust_brightness_parallel(pixels)


start = time.time() # vectorise
result = adjust_brightness_seq(pixels)
time_seq = time.time() - start

start = time.time() # vectorise + parallel
result_parallel = adjust_brightness_parallel(pixels)
time_parallel = time.time() - start

print("Output: ", result[:10])
print()
print("Execution Time: ", time_seq)
print("Execution Time (Parallel): ", time_parallel)
print("Execution Time Diﬀerence : ",time_seq - time_parallel)
print("Speedup: ", time_seq / time_parallel)

Output:  [127 175 255 109 255 255 225 117 255 206]

Execution Time:  0.027329444885253906
Execution Time (Parallel):  0.026324033737182617
Execution Time Diﬀerence :  0.001005411148071289
Speedup:  1.0381936582405737


In [42]:
# Passing Python list instead of NumPy array
pixel_list = list(np.random.randint(0, 256, 10_000_000))

start = time.time()
result = adjust_brightness_seq(pixel_list)
time_list = time.time() - start

print("Output:", result)
print("Execution Time: ", time_list)  # Works... but a lot slower than numpy array

Output: [182 144 218 ... 224 219 184]
Execution Time:  0.7287037372589111


5. Write Python code to generate synthetic training data of 100,000 samples,
10 features and binary labels {-1, +1}. Implement binary logistic regression
using the mathematical formula for gradient descent:

a. Using standard NumPy (without Numba)

b. Using Numba JIT accelera8on

c. Compare correctness and performance.

In [28]:
import numpy as np
from numba import njit

X = np.random.randn(100000, 10)
y = np.random.choice([-1, 1], size=100000)

def train_numpy(X, y, lr=0.01, epochs=100):
    w = np.zeros(X.shape[1])
    for _ in range(epochs):
        z = X @ w
        grad = -(y / (1 + np.exp(y*z))) @ X / len(y)
        w -= lr * grad
    return w

@njit
def train_numba(X, y, lr=0.01, epochs=100):
    w = np.zeros(X.shape[1])
    for _ in range(epochs):
        z = X @ w
        grad = -(y / (1 + np.exp(y * z))) @ X / len(y)
        w -= lr * grad
    return w

start = time.time()
train_numpy(X, y)
print("NumPy:", time.time() - start, "sec")

train_numba(X, y)  # warmup

start = time.time()
train_numba(X, y)
print("Numba:", time.time() - start, "sec")


NumPy: 0.48369550704956055 sec
Numba: 0.3160982131958008 sec


6. Write a CUDA kernel to add two large matrices (A + B = C) of size 1024 X 1024.

In [25]:
@cuda.jit
def mat_add(A, B, C): #CUDA kernel
    i, j = cuda.grid(2)
    if i < A.shape[0] and j < A.shape[1]:
        C[i, j] = A[i, j] + B[i, j]

N = 1024
A = np.random.rand(N, N)
B = np.random.rand(N, N)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array((N, N))

threads = (16, 16)
blocks = (N//16, N//16)

mat_add[blocks, threads](d_A, d_B, d_C)

C = d_C.copy_to_host()

print(f"A: {A}")
print(f"B: {B}")
print()
print("Matrix addition using CUDA kernel")
print(f"C = A + B: {C}")

A: [[0.68524311 0.37530616 0.04944333 ... 0.27101791 0.42158041 0.89936624]
 [0.96329077 0.20885479 0.31893398 ... 0.79926845 0.67606454 0.28635864]
 [0.70413356 0.10025378 0.32997376 ... 0.02777335 0.96619169 0.44649281]
 ...
 [0.26339292 0.20303028 0.12576714 ... 0.4059579  0.36614901 0.24923405]
 [0.98568159 0.19199053 0.99380106 ... 0.77514015 0.10189484 0.51320331]
 [0.5747479  0.66849719 0.30884392 ... 0.00252683 0.28004181 0.81220599]]
B: [[0.06694489 0.33540917 0.77352955 ... 0.47088383 0.15810717 0.45683248]
 [0.62638748 0.6829009  0.71510535 ... 0.43095137 0.71615644 0.51188055]
 [0.36064289 0.25552703 0.99912062 ... 0.83380369 0.76450094 0.58627915]
 ...
 [0.81387635 0.38562806 0.09472345 ... 0.63781497 0.52680261 0.26211956]
 [0.543004   0.75307988 0.86536836 ... 0.28689411 0.13928609 0.27662481]
 [0.66577639 0.73518932 0.60330596 ... 0.35126239 0.69890256 0.9230615 ]]

Matrix addition using CUDA kernel
C = A + B: [[0.752188   0.71071532 0.82297288 ... 0.74190174 0.57968758